# WAXAL-NET fine-tune: ewe × whisper_tiny

Trains on WAXAL ewe, evaluates on the WAXAL test split.

Needs GPU T4 x2, internet, and the `HF_TOKEN` secret (a write token) so checkpoints survive the 12 h session limit. If a run stops at the deadline, run the notebook again to resume.

In [ ]:
LANGUAGE = "ewe"
MODEL = "whisper_tiny"
FLEURS = False
TRAIN_HOURS = 10.5

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

SESSION_START = time.time()


def find_repo():
    for depth in range(1, 5):
        for hit in Path("/kaggle/input").glob("*/" * depth + "src/train.py"):
            return hit.parent.parent
    raise FileNotFoundError("Code not found: attach the waxal-asr-code dataset (Add Input).")


REPO = find_repo()
PY = "/tmp/waxal-venv/bin/python"
os.environ.update(HF_HOME="/tmp/hf_cache", HF_DATASETS_DISABLE_PROGRESS_BARS="1", WANDB_DISABLED="true",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as error:
    print(f"HF_TOKEN secret unavailable ({error}); checkpoints will not survive this session.")


def run(*command, check=True):
    """Run a command from the repo root, streaming its output into the notebook log."""
    process = subprocess.Popen(list(command), cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() and check:
        raise RuntimeError(f"{' '.join(map(str, command))} exited with {process.returncode}")
    return process.returncode


print("Code:", REPO)

In [ ]:
run("bash", f"{REPO}/kaggle/setup_env.sh")

In [ ]:
RUN = f"finetune_{LANGUAGE}_{MODEL}"
CONFIG = f"configs/experiment/{RUN}.yaml"
CHECKPOINT = f"/kaggle/working/checkpoints/{RUN}/best_model"
HUB = ["--hub-repo", f"waxal-net-{RUN}"] if os.environ.get("HF_TOKEN") else []
run(PY, "-m", "src.train", "--config", CONFIG, "--deadline", str(SESSION_START + TRAIN_HOURS * 3600), *HUB)

In [ ]:
if Path(CHECKPOINT, "config.json").exists():
    run(PY, "-m", "src.evaluate", "--config", CONFIG, "--checkpoint", CHECKPOINT)
    if FLEURS:
        run(PY, "-m", "src.cross_domain_eval", "--config", f"configs/experiment/cross_domain_{LANGUAGE}_{MODEL}.yaml",
            "--checkpoint", CHECKPOINT)
else:
    print("Training stopped at the session deadline. Run this notebook again to resume from the last checkpoint.")

In [ ]:
import json, shutil

for stage in ("finetune", "cross_domain"):
    path = Path(f"/kaggle/working/outputs/{stage}_{LANGUAGE}_{MODEL}/metrics.json")
    if path.exists():
        metrics = json.loads(path.read_text())
        print(f"{stage}: WER {metrics['wer_percent']:.1f}% | CER {metrics['cer_percent']:.1f}%")

# Keep the saved notebook output small. With HF_TOKEN every checkpoint and the
# final model are already on the Hub; without it only best_model is kept.
checkpoints = Path(f"/kaggle/working/checkpoints/{RUN}")
if os.environ.get("HF_TOKEN"):
    shutil.rmtree(checkpoints, ignore_errors=True)
else:
    for intermediate in checkpoints.glob("checkpoint-*"):
        shutil.rmtree(intermediate)